In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load and process your data
X_train = pd.read_csv('Datasets/cleaned_X_train2.csv')
y_train = pd.read_csv('Datasets/y_train.csv')

# Convert y_train to a single column of labels (use idxmax if multiple columns exist)
y_train = y_train.idxmax(axis=1)

# Encode the labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Scale the X_train dataset
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Create a SMOTE object
smote = SMOTE(random_state=42)

# Apply SMOTE to balance the training data
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train_encoded)

# Apply PCA
pca = PCA(n_components=0.2)  # Retain 20% of the variance
X_train_pca = pca.fit_transform(X_train_balanced)

# Split the data into training and validation sets
X_train_final, X_val, y_train_final, y_val = train_test_split(X_train_pca, y_train_balanced, test_size=0.2, random_state=42)

# Load the test set
X_test = pd.read_csv('Datasets/cleaned_X_test2.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

# Impute missing values
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Convert y_test to a single column of labels
y_test = y_test.idxmax(axis=1)

# Encode the test labels the same way as the training labels
y_test_encoded = label_encoder.transform(y_test)

# Scale the X_test dataset using the same scaler as X_train
X_test_scaled = scaler.transform(X_test)

# Apply the same PCA transformation to the test data
X_test_pca = pca.transform(X_test_scaled)

# Initialize the XGBoost classifier
xgb_model = XGBClassifier(random_state=42, objective='multi:softmax', num_class=10, early_stopping_rounds=10)

# Define a smaller search space for hyperparameters
param_dist = {
    'learning_rate': np.linspace(0.01, 0.1, 10),
    'n_estimators': [50, 100],  # Reduce the number of trees to prevent overfitting
    'max_depth': [3, 5],  # Limit tree depth
    'subsample': [0.7, 0.8],  # Reduce subsample to prevent overfitting
    'colsample_bytree': [0.7, 0.8],  # Reduce colsample_bytree to prevent overfitting
}

# Use RandomizedSearchCV to optimize hyperparameters
random_search = RandomizedSearchCV(xgb_model, param_distributions=param_dist, n_iter=5, cv=3, scoring='accuracy', random_state=42)

# Train the model with early stopping
random_search.fit(X_train_final, y_train_final, eval_set=[(X_val, y_val)], verbose=False)

# Display the best hyperparameters
print("Best hyperparameters found by RandomizedSearchCV:")
print(random_search.best_params_)

# Predict using the best model on the validation set
y_pred = random_search.best_estimator_.predict(X_val)

# Evaluate on the validation set
accuracy = accuracy_score(y_val, y_pred)
print(f'Accuracy on validation set: {accuracy:.4f}')
print("\nClassification Report on validation set:")
print(classification_report(y_val, y_pred))

# Make predictions on the test set using the best model
y_test_pred = random_search.best_estimator_.predict(X_test_pca)

# Evaluate on the test set
test_accuracy = accuracy_score(y_test_encoded, y_test_pred)
print(f'Test Accuracy: {test_accuracy:.4f}')
print("\nTest Classification Report:")
print(classification_report(y_test_encoded, y_test_pred))


/home/iyoas/.local/lib/python3.10/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Best hyperparameters found by RandomizedSearchCV:
{'subsample': 0.8, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.07, 'colsample_bytree': 0.7}
Accuracy on validation set: 0.4919

Classification Report on validation set:
              precision    recall  f1-score   support

           0       0.43      0.46      0.44        13
           1       0.64      0.56      0.60        16
           2       0.23      0.60      0.33         5
           3       0.32      0.55      0.40        11
           4       0.75      0.46      0.57        13
           5       0.50      0.46      0.48        13
           6       0.30      0.25      0.27        12
           7       0.50      0.45      0.48        11
           8       0.67      0.50      0.57        16
           9       0.75      0.64      0.69        14

    accuracy                           0.49       124
   macro avg       0.51      0.49      0.48       124
weighted avg       0.54      0.49      0.50       124

Test Accur